In [0]:
# Retail Sales Lakehouse - Gold Layer
# Load Silver Delta tables

customers = spark.table("workspace.default.silver_customers")
products = spark.table("workspace.default.silver_products")
orders = spark.table("workspace.default.silver_orders")
order_items = spark.table("workspace.default.silver_order_items")

print("Silver tables loaded successfully")
print("Customers:", customers.count())
print("Products:", products.count())
print("Orders:", orders.count())
print("Order Items:", order_items.count())


In [0]:
from pyspark.sql.functions import col

fact_sales = (
    order_items
    .join(orders, on="order_id", how="inner")
    .join(products, on="product_id", how="inner")
    .join(customers, on="customer_id", how="inner")
    .withColumn("revenue", col("quantity") * col("unit_price"))
    .select(
        "order_item_id",
        "order_id",
        "order_date",
        "customer_id",
        "customer_name",
        "city",
        "product_id",
        "product_name",
        "category",
        "quantity",
        "unit_price",
        "revenue",
        "order_status",
        "sales_channel"
    )
)

display(fact_sales)

In [0]:
completed_sales = (
    fact_sales
    .filter(col("order_status") == "Completed")
)

display(completed_sales)

In [0]:
completed_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_fact_sales")

In [0]:
display(spark.table("workspace.default.gold_fact_sales"))

In [0]:
from pyspark.sql.functions import sum as spark_sum

total_revenue = (
    completed_sales
    .agg(
        spark_sum("revenue").alias("total_revenue")
    )
)

display(total_revenue)

In [0]:
from pyspark.sql.functions import date_format

monthly_sales = (
    completed_sales
    .withColumn("sales_month", date_format(col("order_date"), "yyyy-MM"))
    .groupBy("sales_month")
    .agg(
        spark_sum("revenue").alias("monthly_revenue")
    )
    .orderBy("sales_month")
)

display(monthly_sales)

In [0]:
top_products = (
    completed_sales
    .groupBy("product_id", "product_name", "category")
    .agg(
        spark_sum("quantity").alias("total_quantity_sold"),
        spark_sum("revenue").alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)

display(top_products)

In [0]:
top_customers = (
    completed_sales
    .groupBy("customer_id", "customer_name", "city")
    .agg(
        spark_sum("revenue").alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)

display(top_customers)

In [0]:
channel_performance = (
    completed_sales
    .groupBy("sales_channel")
    .agg(
        spark_sum("revenue").alias("total_revenue"),
        spark_sum("quantity").alias("total_quantity_sold")
    )
    .orderBy(col("total_revenue").desc())
)

display(channel_performance)

In [0]:
monthly_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_monthly_sales")

top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_top_products")

top_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_top_customers")

channel_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_channel_performance")

In [0]:
gold_tables = [
    "workspace.default.gold_fact_sales",
    "workspace.default.gold_monthly_sales",
    "workspace.default.gold_top_products",
    "workspace.default.gold_top_customers",
    "workspace.default.gold_channel_performance"
]

for table in gold_tables:
    df = spark.table(table)
    print(f"{table} -> {df.count()} rows")